In [1]:
import io, math, time, random
import polars as pl
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torch_geometric.nn import SAGEConv

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [9]:
BASE = 'https://huggingface.co/datasets/najrum/cf-interactions/resolve/main/'

def fetch(path):
    r = requests.get(BASE + path, timeout=120); r.raise_for_status()
    return io.BytesIO(r.content)

train    = pl.read_parquet(fetch('interactions/train.parquet'))
test     = pl.read_parquet(fetch('interactions/test.parquet'))
problems = pl.read_parquet(fetch('problems.parquet'))
users    = pl.read_csv(fetch('users.csv'))

train = train.with_columns(pl.col('final_rating').fill_null(0.0))
test  = test.with_columns(pl.col('final_rating').fill_null(0.0))

print(train.shape, test.shape, problems.shape, users.shape)
train.head(3)

(11945545, 6) (1327283, 6) (35839, 4) (140194, 3)


user_index,problem_index,solved,tries,final_rating,experience_years
i64,i64,bool,i64,f64,f64
41881,31,true,2,1173.0,0.717532
9637,3567,true,2,700.0,0.409326
119707,589,true,1,1700.0,2.417739


In [5]:
TAG_DIM  = 32
EMB_DIM  = 64

all_tags   = sorted({t for row in problems['problem_tags'].to_list() for t in (row or [])})
n_tags     = max(all_tags) + 1 if all_tags else 1
n_problems = int(problems['problem_index'].max()) + 1 # type: ignore
n_users    = int(train['user_index'].max()) + 1 # type: ignore

tag_emb_table = nn.Embedding(n_tags, TAG_DIM).to(DEVICE)

def make_problem_features(problems_df: pl.DataFrame, tag_table: nn.Embedding) -> Tensor:
    dev = tag_table.weight.device
    ratings = torch.zeros(n_problems, device=dev)
    for row in problems_df.iter_rows(named=True):
        r = row['problem_rating']
        ratings[row['problem_index']] = float(r) / 4000.0 if r else 0.0

    tag_vecs = torch.zeros(n_problems, TAG_DIM, device=dev)
    counts   = torch.zeros(n_problems, 1, device=dev)
    for row in problems_df.iter_rows(named=True):
        tags = row['problem_tags'] or []
        if not tags: continue
        pi = row['problem_index']
        with torch.no_grad():
            vecs = tag_table(torch.tensor(tags, device=dev))
        tag_vecs[pi] += vecs.sum(0)
        counts[pi]   += len(tags)
    tag_vecs = tag_vecs / counts.clamp(min=1)

    return torch.cat([ratings.unsqueeze(1), tag_vecs], dim=1)

def make_edge_features(df: pl.DataFrame) -> Tensor:
    solved  = torch.tensor(df['solved'].to_list(), dtype=torch.float32)
    tries   = torch.log1p(torch.tensor(df['tries'].to_list(), dtype=torch.float32)) / math.log(100)
    rating  = torch.tensor(df['final_rating'].fill_null(0).to_list(), dtype=torch.float32).clamp(0, 4000) / 4000
    exp     = torch.tensor(df['experience_years'].to_list(), dtype=torch.float32).clamp(0, 10) / 10
    return torch.stack([solved, tries, rating, exp], dim=1)

train_edge_feat = make_edge_features(train).to(DEVICE)
print('problem_feat shape:', make_problem_features(problems, tag_emb_table).shape)
print('edge_feat shape:   ', train_edge_feat.shape)

problem_feat shape: torch.Size([35839, 33])
edge_feat shape:    torch.Size([11945545, 4])


In [6]:
u_idx = torch.tensor(train['user_index'].to_list(),   dtype=torch.long)
p_idx = torch.tensor(train['problem_index'].to_list(), dtype=torch.long)

edge_index = torch.stack([
    torch.cat([u_idx, p_idx + n_users]), # two index types, so we shift problem indices by n_users to avoid overlap
    torch.cat([p_idx + n_users, u_idx]),
], dim=0).to(DEVICE)  # (2, 2E)

train_edge_feat = torch.cat([train_edge_feat, train_edge_feat], dim=0)  # (2E, 4)

print('edge_index:', edge_index.shape)

edge_index: torch.Size([2, 23891090])


In [7]:
class ProblemEncoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.GELU(),
            nn.Linear(out_dim, out_dim),
        )
    def forward(self, x): return self.net(x)


class CFRecommender(nn.Module):
    def __init__(self, n_tags, tag_dim=TAG_DIM, emb_dim=EMB_DIM, n_layers=2):
        super().__init__()
        prob_in = 1 + tag_dim  # rating + tag emb
        edge_in = 4            # solved, tries, rating, exp

        self.tag_emb      = nn.Embedding(n_tags, tag_dim)
        self.prob_encoder = ProblemEncoder(prob_in, emb_dim)

        self.edge_proj = nn.Linear(edge_in, emb_dim)
        self.convs = nn.ModuleList([
            SAGEConv(emb_dim, emb_dim) for _ in range(n_layers)
        ])
        self.user_feat_mlp = nn.Sequential(
            nn.Linear(2, emb_dim), nn.GELU(), nn.Linear(emb_dim, emb_dim)
        )

        self._init()

    def _init(self):
        nn.init.xavier_uniform_(self.tag_emb.weight)

    def problem_features(self, problems_df):
        return make_problem_features(problems_df, self.tag_emb)

    def encode(self, prob_feat, edge_index, user_feat):
        p_emb = self.prob_encoder(prob_feat)
        u_emb = self.user_feat_mlp(user_feat)
        x = torch.cat([u_emb, p_emb], dim=0)
        for conv in self.convs:
            x = F.gelu(conv(x, edge_index))
        return x[:n_users], x[n_users:]

    def forward(self, u_idx, pos_p, neg_p, prob_feat, edge_index, edge_feat, user_feat):
        u_emb, p_emb = self.encode(prob_feat, edge_index, user_feat)
        pos_score = (u_emb[u_idx] * p_emb[pos_p]).sum(-1)
        neg_score = (u_emb[u_idx] * p_emb[neg_p]).sum(-1)
        return -F.logsigmoid(pos_score - neg_score).mean()

model = CFRecommender(n_tags).to(DEVICE)
print(sum(p.numel() for p in model.parameters()), 'params')

28768 params


In [11]:
u = torch.tensor(train['user_index'].to_numpy(), dtype=torch.long)
r = torch.tensor(train['final_rating'].to_numpy(), dtype=torch.float32)
e = torch.tensor(train['experience_years'].to_numpy(), dtype=torch.float32)

user_rating = torch.zeros(n_users, dtype=torch.float32)
user_exp    = torch.zeros(n_users, dtype=torch.float32)
user_counts = torch.zeros(n_users, dtype=torch.float32)

user_rating.index_add_(0, u, r)
user_exp.index_add_(0, u, e)
user_counts.index_add_(0, u, torch.ones_like(r))

user_counts = user_counts.clamp(min=1)

user_feat = torch.stack([
    (user_rating / user_counts).clamp(0, 4000) / 4000,
    (user_exp    / user_counts).clamp(0, 10)   / 10,
], dim=1).to(DEVICE)

print('user_feat:', user_feat.shape)

user_feat: torch.Size([140191, 2])


In [19]:
class BPRDataset(Dataset): # Bayesian Personalized Ranking (BPR) loss dataset
    def __init__(self, df, problems_df, n_problems, neg_ratio=4, hard_frac=0.5):
        solved = df.filter(pl.col('solved'))
        self.users      = torch.tensor(solved['user_index'].to_list(),    dtype=torch.long)
        self.pos_probs  = torch.tensor(solved['problem_index'].to_list(), dtype=torch.long)
        self.u_rating   = torch.tensor(solved['final_rating'].to_list(),  dtype=torch.float32)
        self.n_problems = n_problems
        self.neg_ratio  = neg_ratio
        self.hard_frac  = hard_frac

        pr = torch.zeros(n_problems)
        for row in problems_df.iter_rows(named=True):
            if row['problem_rating']: pr[row['problem_index']] = row['problem_rating']
        self.pr = pr

        self.seen = {}
        for u, p in zip(df['user_index'].to_list(), df['problem_index'].to_list()):
            self.seen.setdefault(u, set()).add(p)

    def __len__(self): return len(self.users) * self.neg_ratio

    def __getitem__(self, idx):
        i = idx // self.neg_ratio
        u, pos, ur, seen = int(self.users[i]), int(self.pos_probs[i]), float(self.u_rating[i]), self.seen.get(int(self.users[i]), set())
        hard = random.random() < self.hard_frac
        for _ in range(20):
            if hard:
                cands = ((self.pr - ur).abs() < 300).nonzero(as_tuple=True)[0]
                neg = int(cands[random.randrange(len(cands))]) if len(cands) else random.randrange(self.n_problems)
            else:
                neg = random.randrange(self.n_problems)
            if neg not in seen and neg != pos: break
        return torch.tensor(u), torch.tensor(pos), torch.tensor(neg)

dataset = BPRDataset(train, problems, n_problems)
loader  = DataLoader(dataset, batch_size=2048, shuffle=True, num_workers=2, pin_memory=True)

In [14]:
@torch.no_grad()
def evaluate(model, test_df, train_df, k=10, max_users=2000):
    model.eval()
    prob_feat = model.problem_features(problems)
    u_emb, p_emb = model.encode(prob_feat, edge_index, train_edge_feat, user_feat)

    gt    = {}
    seen  = {}
    for u, p, s in zip(test_df['user_index'].to_list(), test_df['problem_index'].to_list(), test_df['solved'].to_list()):
        if s: gt.setdefault(u, set()).add(p)
    for u, p in zip(train_df['user_index'].to_list(), train_df['problem_index'].to_list()):
        seen.setdefault(u, set()).add(p)

    eval_users = list(gt.keys())
    random.shuffle(eval_users)
    eval_users = eval_users[:max_users]

    ndcg = recall = 0.0
    for u in eval_users:
        scores = (u_emb[u] @ p_emb.T).cpu()
        for p in seen.get(u, []): scores[p] = -1e9
        topk  = scores.topk(k).indices.tolist()
        rel   = gt[u]
        hits  = [1.0 if p in rel else 0.0 for p in topk]
        dcg   = sum(h / math.log2(i + 2) for i, h in enumerate(hits))
        idcg  = sum(1 / math.log2(i + 2) for i in range(min(len(rel), k)))
        ndcg   += dcg / idcg if idcg else 0.0
        recall += sum(hits) / min(len(rel), k)

    n = len(eval_users)
    return {'ndcg': ndcg / n, 'recall': recall / n}

In [ ]:
N_EPOCHS  = 30
opt       = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
best_ndcg = 0.0

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    t0 = time.time()
    prob_feat = model.problem_features(problems)  # rebuild each epoch (tag_emb updates)
    total_loss = 0

    for u, pos, neg in loader:
        u, pos, neg = u.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        opt.zero_grad()
        loss = model(u, pos, neg, prob_feat, edge_index, train_edge_feat, user_feat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()

    scheduler.step()
    metrics = evaluate(model, test, train)
    mark = ''
    if metrics['ndcg'] > best_ndcg:
        best_ndcg = metrics['ndcg']
        torch.save(model.state_dict(), 'models/gnn/model.pt')
        mark = ' ← best'
    print(f"E{epoch:02d} loss={total_loss/len(loader):.4f}  ndcg@10={metrics['ndcg']:.4f}  recall@10={metrics['recall']:.4f}  [{time.time()-t0:.1f}s]{mark}")

KeyboardInterrupt: 

In [23]:
import requests

@torch.no_grad()
def recommend_new_user(
    history: list[dict],  # [{'problem_index': int, 'solved': bool, 'tries': int, 'final_rating': int, 'experience_years': float}]
    current_rating: float,
    experience_years: float,
    top_k: int = 10,
    exclude_seen: bool = True,
):
    model.eval()
    prob_feat = model.problem_features(problems)

    p_indices = [h['problem_index'] for h in history]
    src = torch.tensor([0] * len(history) + list(range(1, len(history)+1)), dtype=torch.long)
    dst = torch.tensor(list(range(1, len(history)+1)) + [0] * len(history), dtype=torch.long)
    mini_edge_index = torch.stack([src, dst]).to(DEVICE)

    ef = make_edge_features(pl.DataFrame(history)).to(DEVICE)
    ef = torch.cat([ef, ef])

    u_f = torch.tensor([[current_rating / 4000.0, min(experience_years, 10) / 10.0]], device=DEVICE)

    neighbor_feats = prob_feat[torch.tensor(p_indices)]
    neighbor_embs  = model.prob_encoder(neighbor_feats)  # (|history|, D)
    u_init         = model.user_feat_mlp(u_f)            # (1, D)

    x = torch.cat([u_init, neighbor_embs], dim=0)        # (1+|history|, D)
    for conv in model.convs:
        x = F.gelu(conv(x, mini_edge_index))

    u_emb = x[0]                                         # (D,)
    p_emb = model.prob_encoder(prob_feat)                 # (P, D)

    scores = (u_emb @ p_emb.T).cpu()                     # (P,)
    if exclude_seen:
        for p in p_indices: scores[p] = -1e9

    top = scores.topk(top_k)
    return [
        {'problem_index': int(i), 'score': float(s)}
        for i, s in zip(top.indices, top.values)
    ]

def fetch_user_info(handle: str):
    base_url = 'https://codeforces.com/api/'
    
    info_res = requests.get(f'{base_url}user.info?handles={handle}')
    info_res.raise_for_status()
    user_info = info_res.json()['result'][0]
    
    current_rating = user_info.get('rating', 0)
    reg_ts = user_info['registrationTimeSeconds']
    experience_years = max(0, (time.time() - reg_ts) / (365.25 * 24 * 3600))
    
    rating_res = requests.get(f'{base_url}user.rating?handle={handle}')
    rating_res.raise_for_status()
    rating_history = rating_res.json().get('result', [])
    
    status_res = requests.get(f'{base_url}user.status?handle={handle}')
    status_res.raise_for_status()
    submissions = status_res.json().get('result', [])
    
    handled_verdicts = {
        "FAILED", "OK", "COMPILATION_ERROR", "RUNTIME_ERROR", 
        "WRONG_ANSWER", "TIME_LIMIT_EXCEEDED", "MEMORY_LIMIT_EXCEEDED", 
        "IDLENESS_LIMIT_EXCEEDED", "SECURITY_VIOLATED", "REJECTED"
    }
    
    problem_subs = {}
    for sub in submissions:
        if sub.get("verdict") not in handled_verdicts:
            continue
        if sub["problem"]["type"] != "PROGRAMMING":
            continue
            
        p = sub["problem"]
        pid = f"{p.get('contestId', '')}.{p.get('index', '')}"
        problem_subs.setdefault(pid, []).append(sub)
        
    history = []
    for pid, subs in problem_subs.items():
        dataset_row = problems.filter(pl.col('problem_id') == pid)
        if dataset_row.is_empty():
            continue
        problem_index = dataset_row[0, 'problem_index']

        subs_chrono = list(reversed(subs))
        verdicts = [s.get("verdict") for s in subs_chrono]
        
        solved = "OK" in verdicts
        first_ok_idx = verdicts.index("OK") if solved else None
        tries = (first_ok_idx + 1) if solved else len(subs_chrono) # type: ignore
        
        decisive_sub = subs_chrono[first_ok_idx] if solved else subs_chrono[-1] # type: ignore
        decisive_ts = decisive_sub.get("creationTimeSeconds", reg_ts)
        
        historical_rating = None
        best_ts = -1
        for entry in rating_history:
            t_hist = entry.get("ratingUpdateTimeSeconds", 0)
            if t_hist <= decisive_ts and t_hist > best_ts:
                best_ts = t_hist
                historical_rating = entry.get("newRating")
        
        history.append({
            "problem_index": problem_index,
            "solved": solved,
            "tries": tries,
            "final_rating": historical_rating,
            "experience_years": (decisive_ts - reg_ts) / (365.25 * 24 * 3600)
        })
        
    return history, current_rating, experience_years

recs = recommend_new_user(*fetch_user_info('tourist'))
for r in recs[:5]:
    problem_index = r['problem_index']
    problem_row = problems.filter(pl.col('problem_index') == problem_index)
    problem_id = problem_row[0, 'problem_id']
    contest_id, index = problem_id.split('.')
    print(f"Recommended problem: https://codeforces.com/contest/{contest_id}/problem/{index} with score {r['score']:.4f}")

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


Recommended problem: https://codeforces.com/contest/2220/problem/B with score 4.5372
Recommended problem: https://codeforces.com/contest/391/problem/C2 with score 4.5372
Recommended problem: https://codeforces.com/contest/391/problem/F2 with score 4.5372
Recommended problem: https://codeforces.com/contest/1319/problem/C with score 3.9611
Recommended problem: https://codeforces.com/contest/392/problem/A with score 3.5461
